# Pipeline Pelatihan Model PERISAI (Versi Master)

Notebook ini memenuhi standar pengembangan Deep Learning lanjutan dengan spesifikasi:
1. **Arsitektur**: TensorFlow Functional API.
2. **Komponen Kustom**: `AdvancedDenseLayer` (Custom Layer).
3. **Training Loop**: Custom Loop menggunakan `tf.GradientTape`.
4. **Logging**: Integrasi TensorBoard.
5. **Target Performa**: Akurasi ≥ 85% dan MAE ≤ 0.02.
6. **Parameter**: 10 parameter kesehatan sesuai ketentuan.

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1778844344.945333   25712 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778844344.978154   25712 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778844345.814137   25712 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


In [2]:
# 1. Load Data (Master Synthetic)
df = pd.read_csv('../data/perisai_synthetic_master.csv')

FEATURES = ['Usia', 'BMI', 'Pola_Makan', 'Aktivitas_Fisik', 'Kualitas_Tidur', 
            'Tingkat_Stres', 'Riwayat_Keluarga', 'Status_Merokok', 
            'Konsumsi_Alkohol', 'Riwayat_Penyakit']
TARGETS  = ['Diabetes_binary', 'HighBP', 'HighChol']

X = df[FEATURES].values.astype('float32')
y = df[TARGETS].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Dataset Master Siap! Total data: {len(df)}")

Dataset Master Siap! Total data: 50000


In [3]:
# 2. Custom Component: AdvancedDenseLayer
class AdvancedDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super(AdvancedDenseLayer, self).__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="glorot_uniform",
            trainable=True,
            name="kernel"
        )
        self.b = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True,
            name="bias"
        )

    def call(self, inputs):
        z = tf.matmul(inputs, self.w) + self.b
        if self.activation is not None:
            return self.activation(z)
        return z

In [4]:
# 3. Functional API Model
def build_functional_model(input_dim):
    inputs = tf.keras.Input(shape=(input_dim,), name="health_params")
    
    # Menggunakan Custom Layer
    x = AdvancedDenseLayer(256, activation="relu")(inputs)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = AdvancedDenseLayer(64, activation="relu")(x)
    
    outputs = tf.keras.layers.Dense(3, activation="sigmoid", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="PERISAI_Master_Model")
    return model

model = build_functional_model(len(FEATURES))
model.summary()

E0000 00:00:1778844348.369709   25712 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1778844348.369732   25712 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1778844348.369739   25712 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1778844348.369747   25712 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1778844348.369748   25712 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1778844348.369750   25712 cuda_diagnostics.cc:183] hostname: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1778844348.369901   25712 cuda_diagnostics.cc:190] libcuda reported version is: 580.142.0
I0000 00:00:1778844348.369913   25

Model: "PERISAI_Master_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ health_params (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ advanced_dense_layer            │ (None, 256)            │         2,816 │
│ (AdvancedDenseLayer)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ advanced_dense_layer_1          │ (None, 64)             │         8,256 │
│ (AdvancedDenseLayer)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,163 (172.51 KB)

 Trainable params: 44,163 (172.51 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 4. Custom Training Loop (Side Quest Requirement)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.BinaryCrossentropy()

# Metrics
train_acc_metric = tf.keras.metrics.BinaryAccuracy()
val_acc_metric = tf.keras.metrics.BinaryAccuracy()
mae_metric = tf.keras.metrics.MeanAbsoluteError()

# TensorBoard Setup
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = "../logs/gradient_tape/" + current_time
summary_writer = tf.summary.create_file_writer(log_dir)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss_value = loss_fn(y, logits)
    grads = tape.gradient(loss_value, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    train_acc_metric.update_state(y, logits)
    mae_metric.update_state(y, logits)
    return loss_value

@tf.function
def test_step(x, y):
    val_logits = model(x, training=False)
    val_acc_metric.update_state(y, val_logits)

epochs = 30
batch_size = 64
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)

print("Memulai Custom Training Loop...")
for epoch in range(epochs):
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        loss_val = train_step(x_batch_train, y_batch_train)
        
    # Log metrics to TensorBoard
    train_acc = train_acc_metric.result()
    current_mae = mae_metric.result()
    
    with summary_writer.as_default():
        tf.summary.scalar('loss', loss_val, step=epoch)
        tf.summary.scalar('accuracy', train_acc, step=epoch)
        tf.summary.scalar('mae', current_mae, step=epoch)
    
    for x_batch_val, y_batch_val in val_dataset:
        test_step(x_batch_val, y_batch_val)
    
    val_acc = val_acc_metric.result()
    if epoch % 5 == 0 or epoch == epochs-1:
        print(f"Epoch {epoch}: Loss: {loss_val:.4f}, Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, MAE: {current_mae:.4f}")
    
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    mae_metric.reset_state()

print("Training Selesai!")

Memulai Custom Training Loop...
Epoch 0: Loss: 0.0438, Acc: 0.9595, Val Acc: 0.9862, MAE: 0.0678
Epoch 5: Loss: 0.0168, Acc: 0.9920, Val Acc: 0.9904, MAE: 0.0118
Epoch 10: Loss: 0.0096, Acc: 0.9935, Val Acc: 0.9909, MAE: 0.0092
Epoch 15: Loss: 0.0043, Acc: 0.9944, Val Acc: 0.9915, MAE: 0.0077
Epoch 20: Loss: 0.0078, Acc: 0.9945, Val Acc: 0.9940, MAE: 0.0072
Epoch 25: Loss: 0.0125, Acc: 0.9947, Val Acc: 0.9938, MAE: 0.0069
Epoch 29: Loss: 0.0069, Acc: 0.9951, Val Acc: 0.9940, MAE: 0.0066
Training Selesai!


In [6]:
# 5. Evaluasi & Simpan Model
test_logits = model(X_test, training=False)
test_acc = tf.keras.metrics.BinaryAccuracy()(y_test, test_logits)
test_mae = tf.keras.metrics.MeanAbsoluteError()(y_test, test_logits)

print('\n' + '='*40)
print(f"FINAL TEST ACCURACY: {test_acc.numpy()*100:.2f}%")
print(f"FINAL TEST MAE     : {test_mae.numpy():.4f}")
print('='*40)

if test_acc >= 0.85 and test_mae <= 0.02:
    print("Target Performa Tercapai! (✅)")
else:
    print("Target Performa Belum Tercapai. (❌)")

model.save('../perisai_model_production.keras')
print("Model disimpan ke ../perisai_model_production.keras")


FINAL TEST ACCURACY: 99.44%
FINAL TEST MAE     : 0.0067
Target Performa Tercapai! (✅)
Model disimpan ke ../perisai_model_production.keras


In [22]:
# 6. Simple Inference Code
idx = int(np.random.randint(len(X_test)))
sample_X = X_test[idx:idx+1]
sample_y_true = y_test[idx]

predictions = model.predict(sample_X, verbose=0)[0]
pred_labels = (predictions > 0.5).astype(int)

feature_names = ['Usia', 'BMI', 'Pola Makan', 'Aktifitas Fisik', 'Kualitas Tidur', 
                 'Tingkat Stres', 'Riwayat Keluarga', 'Status Merokok', 
                 'Konsumsi Alkohol', 'Riwayat Penyakit']
target_names = ['Diabetes', 'Hipertensi', 'Kolesterol']

print('\n' + '='*50)
print('   SISTEM INFERENSI PERISAI')
print('='*50)

print('\n[INPUT PASIEN]')
for name, val in zip(feature_names, sample_X[0]):
    print(f" - {name:20}: {val:.3f} (Scaled)")

print('\n[HASIL DIAGNOSA]')
for name, prob, label, true in zip(target_names, predictions, pred_labels, sample_y_true):
    status = "POSITIF" if label == 1 else "NEGATIF"
    actual = "POSITIF" if int(true) == 1 else "NEGATIF"
    match  = "✅" if label == int(true) else "❌"
    print(f" - {name:15}: {status:8} (Prob: {prob:.3f}) | Data Asli: {actual} {match}")

print('\n' + '='*50)


   SISTEM INFERENSI PERISAI

[INPUT PASIEN]
 - Usia                : 0.990 (Scaled)
 - BMI                 : 1.106 (Scaled)
 - Pola Makan          : 0.562 (Scaled)
 - Aktifitas Fisik     : 1.295 (Scaled)
 - Kualitas Tidur      : 1.721 (Scaled)
 - Tingkat Stres       : 1.246 (Scaled)
 - Riwayat Keluarga    : -0.653 (Scaled)
 - Status Merokok      : 1.233 (Scaled)
 - Konsumsi Alkohol    : -0.047 (Scaled)
 - Riwayat Penyakit    : -0.501 (Scaled)

[HASIL DIAGNOSA]
 - Diabetes       : POSITIF  (Prob: 1.000) | Data Asli: POSITIF ✅
 - Hipertensi     : POSITIF  (Prob: 1.000) | Data Asli: POSITIF ✅
 - Kolesterol     : POSITIF  (Prob: 1.000) | Data Asli: POSITIF ✅

